# 第2章 第10〜12回 解決策を実装する（画像認識・画像生成）

## はじめに

### 前回の振り返り

前回までの授業で、あなたが取り組む課題と、課題解決に成功したといえる条件を決めました。

- ⑤ 誰の、どんな困りごとを解決するか
- ⑥ 仮説
- ⑦ 成功指標
- ⑧ テスト例（4件）
- ⑩ 最小限の機能

ここからは、画像認識や画像生成などを使い、実際に課題解決のためのプログラムを作っていきます。

### このノートブックでやること

今回（第10回）は、⑩で決めた最小限の機能を作り、動かしてみます。\
次回（第11回）は、出力を比べて改善するところを決めます。\
第12回は、⑧のテスト例で結果を確かめます。

演習を通して、次の3つを説明できるようになることを目指します。

1. 見本を生成AIに改変してもらい、自分の課題に合うプログラムを作る流れを説明する。
2. 期待する結果と実際の結果を比較し、改善する点を考える。
3. 成功指標に照らして、困りごとの解決につながったかを確かめる。

**このノートブックは第10〜12回で続けて使います。**

## 1. 最小限の機能を作る（第10回）

ここでは、第8〜9回で決めた⑩の最小限の機能を、ひな型（見本）をもとに作ります。\
作るもののタイプを選び、見本を動かしたあと、生成AIに見本を改変してもらいます。

### 1-1. 解説：作るもののタイプを選ぶ

第8回で紹介した画像認識・画像生成などの機能を、7つのタイプに分けました。\
CNN（Convolutional Neural Network／畳み込みニューラルネットワーク）は、画像の特徴を学習するAIです。LoRA（Low-Rank Adaptation／少ない追加パラメータでモデルを調整する方法）は、画像生成の絵柄などを変えるために使います。\
タイプごとに、改変のもとになるひな型（見本）があります。第8回の③「入力・させること・出力」に近いタイプを1つ選びましょう。

|タイプNo| 作れるもの | 見本のプログラム | 入力 → 出力 |
| :--- | :--- | :--- | :--- |
| 1 | 画像の種類を予測する | 衣類などを分類する | 画像 → 予測した種類 |
| 2 | 似ている画像を探す | 画像の特徴を比べる | 画像 → 似ている画像 |
| 3 | 複数の画像を仕分ける | 画像をまとめて分類する | 複数の画像 → 分類結果 |
| 4 | 指示文から画像を作る | 学校のお知らせ用の挿絵 | 指示文 → 画像 |
| 5 | 絵柄を揃えて画像を作る | 水彩画風の挿絵 | 指示文 → 同じ画風の画像 |
| 6 | 画像に文字や余白を加える | 掲示用の画像を作る | 題名 → 文字入り画像 |
| 7 | 決まった手順で画像を加工する（AIを使わない） | 画像を同じ大きさに揃える | 大きさ → 加工画像 |

まずは自分に合うタイプを選び、詳しい条件は見本を動かしてから考えましょう。

#### 参考：利用できるデータ・LoRA

次のデータには画像と、種類を示すラベルなどが含まれています。使い道は自分で考えてかまいません。

| 名前 | データの内容 | この見本での利用 | 出典・利用条件 |
| :--- | :--- | :--- | :--- |
| Fashion-MNIST | 衣類10種類の白黒画像（28×28画素） | すぐ使える | [公式](https://github.com/zalandoresearch/fashion-mnist)／MIT |
| Kuzushiji-MNIST | くずし字10種類の白黒画像（28×28画素） | すぐ使える | [公式](https://github.com/rois-codh/kmnist)／CC BY-SA 4.0 |
| Quick, Draw! | 手描きの絵345種類 | 先生の準備が必要 | [公式](https://github.com/googlecreativelab/quickdraw-dataset)／CC BY 4.0 |
| Recyclable Waste | 資源ごみ5種類の写真 | 先生の準備が必要 | [配布元](https://data.mendeley.com/datasets/h5pxbsdz4m/1) |
| GeoSetEnnea | 作図用具9種類の写真と位置ラベル | 先生の準備が必要 | [配布元](https://data.mendeley.com/datasets/fg97mhsdbr/1) |

画像生成では、[水彩画風](https://huggingface.co/ostris/watercolor_style_lora_sdxl)などのLoRAが選べます。使うモデルと利用条件は先生の案内を確認しましょう。

**今回の見本で直接読み込める画像データは、最初の2種類です。** それ以外を使う場合は、先生から対応するデータと見本を受け取ってください。

 > **「作るもののタイプ」の記入例**
 >
 > 例1（リユース用の衣類画像を整理する）：画像を種類ごとに予測する
 > 例2（委員会のお知らせに挿絵を付ける）：絵柄を揃えて画像を作る
 >
 > **使い道を決めずに、提供データの内容から課題を考えてもかまいません。**

---
**作るもののタイプ**<font color="red">＜入力＞</font>

- 選んだタイプ：

---

### 1-2. 実行：画像データやAIを準備する

プログラムを動かすために、準備のセルを実行しましょう。Google Colaboratory（Colab）は、ブラウザーでノートブックを実行できる環境です。\
選んだタイプによって使う準備セルが違います。**準備用のコードは書きかえず**、必要なセルの実行ボタンを押してください。

| 選んだタイプ | 実行するセル |
| :--- | :--- |
| 1・2・3（画像認識） | 共通の準備 → CNNの準備 |
| 4・5（画像生成） | 共通の準備 → 画像生成の準備 |
| 6・7（AIを使わない） | 共通の準備 |

画像生成ではGPU（画像処理などを高速に行う装置）が必要です。ColabでGPUが使えない場合は先生に伝えましょう。

#### **共通の準備**
該当タイプ：1〜7（全員）

最初に次のプログラム例を実行しましょう。画像の表示やテストに使う準備をします。

**全員：共通の準備を実行する**

##### プログラム例

「共通の準備ができました。」と表示されたら完了です。

In [ ]:
# 共通の準備
import os, sys, math, json, re
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageOps
import matplotlib.pyplot as plt
from IPython.display import display
import torch

OUTPUT_DIR = Path('/content/cv_lesson_results') if Path('/content').exists() else Path('cv_lesson_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def 表示する(結果):
    """文字・画像・辞書を表示します。"""
    if isinstance(結果, Image.Image):
        display(結果)
    elif isinstance(結果, dict):
        for 名前, 値 in 結果.items():
            if isinstance(値, Image.Image):
                print(f'【{名前}】')
                display(値)
            elif isinstance(値, list) and 値 and isinstance(値[0], Image.Image):
                print(f'【{名前}】')
                for 画像 in 値[:6]: display(画像)
            else:
                print(f'{名前}: {値}')
    else:
        print(結果)

def 保存して表示(結果, 回, 番号):
    """同じ入力で改善前後を比べられるように結果を別名保存します。"""
    保存先 = OUTPUT_DIR / f'{回}_test{番号}'
    if isinstance(結果, Image.Image):
        結果.save(保存先.with_suffix('.png'))
    elif isinstance(結果, dict):
        文字 = {}
        for キー, 値 in 結果.items():
            if isinstance(値, Image.Image):
                値.save(OUTPUT_DIR / f'{回}_test{番号}_{キー}.png')
            else:
                文字[キー] = str(値)
        保存先.with_suffix('.json').write_text(json.dumps(文字, ensure_ascii=False, indent=2), encoding='utf-8')
    else:
        保存先.with_suffix('.txt').write_text(str(結果), encoding='utf-8')
    表示する(結果)

def テスト実行(番号, 回='第10回'):
    """入力→実行→保存。エラーは未実施・想定外として記録できます。"""
    if 'run_task' not in globals():
        print('先に1-4で選んだ見本、または自分のプログラムを実行してください。')
        return
    try:
        入力 = [テスト1, テスト2, テスト3, テスト4][番号 - 1]
        print(f'{回} / テスト{番号} / 入力: {入力!r}')
        結果 = run_task(入力.strip())
        保存して表示(結果, 回, 番号)
    except Exception as e:
        print(f'【未実施・想定外】{type(e).__name__}: {e}')
        print('できなかった原因も開発ログに記録しましょう。')

print('共通の準備ができました。')
print('GPU利用可能:', torch.cuda.is_available())

#####

**CNNで画像を認識する場合**

タイプ1〜3を選んだ人は、下のプログラム例を実行します。衣類3種類の画像を使って、小さなCNNを学習します。

初めて実行するときは画像をダウンロードします。学習した結果が表示されたら完了です。
くずし字を使いたい場合は、プログラム例の `データセット名` を `kmnist` に変更できます。

##### プログラム例

In [ ]:
# CNNの準備

import torchvision
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

データセット名 = 'fashion_mnist'  # 興味に合わせて 'kmnist' に変えてもよい
使う種類ID = (0, 1, 8)          # fashion_mnist: Tシャツ類・ズボン・バッグ
各種類の訓練枚数 = 100          # CPUで軽く試すための初期設定
学習回数 = 2

カテゴリ名一覧 = {
    'fashion_mnist': ['Tシャツ類','ズボン','プルオーバー','ワンピース','コート','サンダル','シャツ','スニーカー','バッグ','ブーツ'],
    'kmnist': ['お','き','す','つ','な','は','ま','や','れ','を'],
}

class 小さなCNN(nn.Module):
    def __init__(self, クラス数):
        super().__init__()
        self.特徴抽出 = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4)))
        self.分類 = nn.Linear(16 * 4 * 4, クラス数)
    def features(self, x):
        return self.特徴抽出(x).reshape(x.shape[0], -1)
    def forward(self, x):
        return self.分類(self.features(x))

CNN_DATA = None

def CNNを準備(dataset_name='fashion_mnist', class_ids=(0,1,8), per_class=100, epochs=2, seed=42):
    global CNN_DATA
    if dataset_name not in ('fashion_mnist','kmnist'):
        raise ValueError('対応データは fashion_mnist と kmnist です。')
    torch.manual_seed(seed)
    dataset_class = torchvision.datasets.FashionMNIST if dataset_name == 'fashion_mnist' else torchvision.datasets.KMNIST
    root = str(OUTPUT_DIR / 'dataset_cache')
    original_train = dataset_class(root=root, train=True, download=True)
    original_test = dataset_class(root=root, train=False, download=True)
    class_ids = tuple(int(v) for v in class_ids)
    if len(class_ids) < 2 or len(set(class_ids)) != len(class_ids):
        raise ValueError('異なる種類を2つ以上選んでください。')
    if not all(0 <= v < 10 for v in class_ids):
        raise ValueError('使える種類IDは0〜9です。')
    generator = torch.Generator().manual_seed(seed)
    def select(original, count):
        chosen = []
        for class_id in class_ids:
            positions = torch.where(original.targets == class_id)[0]
            order = positions[torch.randperm(len(positions), generator=generator)]
            chosen.extend(order[:count].tolist())
        images = original.data[chosen].float().unsqueeze(1) / 255.0
        targets = original.targets[chosen]
        mapped = torch.tensor([class_ids.index(int(t)) for t in targets], dtype=torch.long)
        return images, mapped
    X, y = select(original_train, per_class)
    order = torch.randperm(len(y), generator=generator)
    cut = int(len(y) * 0.8)
    X_train, y_train = X[order[:cut]], y[order[:cut]]
    X_val, y_val = X[order[cut:]], y[order[cut:]]
    X_test, y_test = select(original_test, 35)
    model = 小さなCNN(len(class_ids))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    loss_function = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
    for epoch in range(epochs):
        model.train()
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = loss_function(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            acc = (model(X_val).argmax(dim=1) == y_val).float().mean().item()
        print(f'{epoch+1}回目: 検証用データの正解率 {acc:.1%}')
    CNN_DATA = dict(model=model, X_test=X_test, y_test=y_test,
                    labels=[カテゴリ名一覧[dataset_name][v] for v in class_ids],
                    name=dataset_name, X_train=X_train, y_train=y_train)
    with torch.no_grad():
        model.eval()
        final_acc = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
    print(f'テスト用画像 {len(y_test)}枚 / 最終正解率 {final_acc:.1%}（学習データには未使用）')
    return CNN_DATA

def 入力画像(入力):
    if isinstance(入力, Image.Image): return 入力.convert('L').resize((28, 28))
    if isinstance(入力, str) and 入力.startswith('sample:'):
        if CNN_DATA is None: raise RuntimeError('先にCNNの準備セルを実行してください。')
        i = int(入力.split(':', 1)[1])
        if not 0 <= i < len(CNN_DATA['X_test']):
            raise ValueError('サンプル番号が範囲外です。')
        x = CNN_DATA['X_test'][i,0].numpy()
        return Image.fromarray(np.uint8(x * 255), 'L')
    if isinstance(入力, str) and Path(入力).is_file():
        return Image.open(入力).convert('L').resize((28,28))
    raise ValueError('sample:0 などの番号か、Colabに置いた画像ファイルのパスを入力してください。')

CNNを準備(データセット名, 使う種類ID, 各種類の訓練枚数, 学習回数)
print('CNNの準備ができました。例: sample:0 （学習に使っていない画像）')

#####

#### **画像生成の準備**
該当タイプ：4.指示文から画像を作る / 5.絵柄を揃えて画像を作る

画像生成を選んだ人は、次のプログラム例を実行します。実際に画像を作るときに、基盤モデルをダウンロードします。

無料版ColabでGPUが利用できない場合は、先生に相談してください。

In [ ]:
# 画像生成の準備

import importlib.util, subprocess

生成モデル = None
今のLoRA = None
基盤モデルID = 'stabilityai/stable-diffusion-xl-base-1.0'
LoRA一覧 = {
    'watercolor': ('ostris/watercolor_style_lora_sdxl', 'watercolor_v1_sdxl.safetensors'),
    'pixel_art': ('ntc-ai/SDXL-LoRA-slider.pixel-art', 'pixel art.safetensors'),
}

def 画像生成を準備(画風なし=False, lora='watercolor'):
    global 生成モデル, 今のLoRA
    if not torch.cuda.is_available():
        raise RuntimeError('無料ColabでGPUが使えません。先生の生成済みサンプルを使って評価してください。')
    if importlib.util.find_spec('diffusers') is None or importlib.util.find_spec('accelerate') is None:
        print('画像生成の専用ライブラリをインストールします。')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'diffusers>=0.30,<1', 'transformers>=4.44,<6',
                               'accelerate>=0.30,<2', 'safetensors'])
    from diffusers import StableDiffusionXLPipeline
    if 生成モデル is None:
        print('SDXLを初回ダウンロードします（容量が大きく、通信制限時は失敗します）。')
        生成モデル = StableDiffusionXLPipeline.from_pretrained(
            基盤モデルID, torch_dtype=torch.float16, use_safetensors=True)
        生成モデル.enable_model_cpu_offload()  # メモリ節約（代わりに遅くなる場合があります）
        生成モデル.enable_vae_tiling()
    使いたいLoRA = None if 画風なし else lora
    if 今のLoRA != 使いたいLoRA:
        if 今のLoRA is not None:
            生成モデル.unload_lora_weights()
        if 使いたいLoRA is not None:
            if 使いたいLoRA not in LoRA一覧:
                raise ValueError('登録されているLoRA: ' + ', '.join(LoRA一覧))
            モデル名, ファイル名 = LoRA一覧[使いたいLoRA]
            生成モデル.load_lora_weights(モデル名, weight_name=ファイル名, adapter_name='lesson_style')
        今のLoRA = 使いたいLoRA
    return 生成モデル

def 画像を作る(指示文, lora=None, strength=0.7, seed=42):
    if not isinstance(指示文, str) or not 指示文.strip():
        return '指示文が空です。作りたい画像の内容を書いてください。'
    p = 画像生成を準備(画風なし=(lora is None), lora=lora or 'watercolor')
    if lora is not None:
        p.set_adapters(['lesson_style'], adapter_weights=[float(strength)])
    # 比較のためにseed/サイズ/ステップ数を統一します。
    g = torch.Generator(device='cpu').manual_seed(seed)
    return p(prompt=指示文, width=512, height=512,
             num_inference_steps=18, guidance_scale=6.5,
             generator=g).images[0]

print('画像生成の準備用関数を登録しました（まだモデルはダウンロードしていません）。')

#### **AIを使わない画像処理の準備**
該当タイプ：6.画像に文字や余白を加える / 7.決まった手順で画像を加工する

共通の準備だけで動かせます。次の1-3へ進みましょう。

### 1-3. ワーク：自分の課題を書き写す

第8〜9回のノートブックを開き、⑤⑥⑦⑩を下の欄に書き写しましょう。\
これからの3回の授業では、この内容をもとにプログラムを作り、動作を確かめていきます。

---
**自分の課題**<font color="red">＜入力＞</font>

- ⑤ まとめの一文：
- ⑥ 仮説：
- ⑦ 成功指標：
- ⑩ 最小限の機能：
- ⑩ 試す入力と期待する出力：
- 使う画像データ・LoRA（あれば）：

---

### 1-4. 実行：見本を動かす

1-1で選んだタイプの見本のセルを、書きかえずに実行しましょう。実行後、確かめることのとおりに動くか確認します。

> **ヒント**\
> 見本を動かす前に、1-2で自分のタイプに必要な準備を実行しましょう。準備をしていないと、`run_task`が使うデータやモデルが見つからないことがあります。
>
> **実行上の注意**
> 7つすべてを動かす必要はありません。**自分が選んだタイプの見本だけ**実行しましょう。

#### 【見本】1.画像の種類を予測する

確かめること：画像と、予測した衣類の種類が表示されるか。

In [ ]:
# 見本1：分類結果を表示する。準備セルで学習したCNNを利用します。
def run_task(入力):
    画像 = 入力画像(入力)
    x = torch.tensor(np.array(画像), dtype=torch.float32).reshape(1,1,28,28) / 255.0
    モデル = CNN_DATA['model'].eval()
    with torch.no_grad():
        確率 = torch.softmax(モデル(x), dim=1)[0]
    予測番号 = int(確率.argmax())
    結果 = {'入力画像': 画像.resize((168,168)),
            '予測': CNN_DATA['labels'][予測番号],
            '予測の確率（正解保証ではない）': f'{float(確率[予測番号]):.1%}'}
    if isinstance(入力, str) and 入力.startswith('sample:'):
        番号 = int(入力.split(':', 1)[1])
        結果['テスト用の正解ラベル'] = CNN_DATA['labels'][int(CNN_DATA['y_test'][番号])]
    return 結果

表示する(run_task('sample:0'))

#### 【見本】2.似ている画像を探す

確かめること：入力画像に似た画像が3枚表示されるか。

In [ ]:
# 見本2：CNNが取り出す特徴を比べる（同じ画像そのものを返さないようにします）。
def run_task(入力):
    画像 = 入力画像(入力)
    x = torch.tensor(np.array(画像), dtype=torch.float32).reshape(1,1,28,28) / 255.0
    モデル = CNN_DATA['model'].eval()
    with torch.no_grad():
        特徴 = モデル.features(x)
        画像一覧 = torch.cat([CNN_DATA['X_test'][5:15], CNN_DATA['X_test'][40:50], CNN_DATA['X_test'][75:85]])
        候補特徴 = モデル.features(画像一覧)
        類似度 = torch.nn.functional.cosine_similarity(特徴, 候補特徴, dim=1)
        上位 = torch.topk(類似度, k=3).indices.tolist()
    結果 = {'探した画像': 画像.resize((168,168)), '説明': 'CNNの特徴に基づく上位3枚'}
    for 順位, i in enumerate(上位, 1):
        candidate = Image.fromarray(np.uint8(画像一覧[i,0].numpy()*255), 'L')
        結果[f'{順位}番目（類似度 {float(類似度[i]):.2f}）'] = candidate.resize((168,168))
    return 結果

表示する(run_task('sample:0'))

#### 【見本】3.複数の画像を仕分ける

確かめること：3枚の画像について、それぞれ予測した種類が表示されるか。

In [ ]:
# 見本3：複数の写真の分類結果をまとめる。
def run_task(入力):
    if not 入力.startswith('sample:'):
        return 'sample:0,1,2 のように画像番号をカンマ区切りで入力してください。'
    番号リスト = [s.strip() for s in 入力.split(':',1)[1].split(',')]
    if not 1 <= len(番号リスト) <= 12:
        return '一度に扱える画像は1〜12枚です。'
    モデル = CNN_DATA['model'].eval()
    画像 = [入力画像('sample:'+n) for n in 番号リスト]
    X = torch.stack([torch.tensor(np.array(img), dtype=torch.float32).unsqueeze(0)/255 for img in 画像])
    with torch.no_grad():
        予測 = モデル(X).argmax(dim=1).tolist()
    明細 = [f'画像{n}: {CNN_DATA["labels"][p]}' for n, p in zip(番号リスト,予測)]
    return {'分類結果': '／'.join(明細), '処理した枚数': len(明細)}

表示する(run_task('sample:0,40,70'))

#### 【見本】4.指示文から画像を作る

確かめること：学校の図書室を表す挿絵が1枚生成されるか。

In [ ]:
# 見本4：LoRAを使わずに画像を生成します（GPUがある場合だけ）。
def run_task(指示文):
    return 画像を作る(指示文, lora=None, seed=42)

try:
    表示する(run_task('A quiet school library, warm sunlight, simple illustration, no text'))
except Exception as e:
    print('【未実施】画像生成はできませんでした:', type(e).__name__, str(e)[:250])

#### 【見本】5.絵柄を揃えて画像を作る

確かめること：水彩画風の挿絵が生成されるか。

In [ ]:
# 見本5：水彩画風LoRAを使います。生成モデルは1-2で準備した関数から読み込みます。
画風の種類 = 'watercolor'  # 'pixel_art' に変えることもできます
適用の強さ = 0.7           # 0.0〜1.2を目安に、1つずつ変えて比較しましょう

def run_task(指示文):
    return 画像を作る(指示文, lora=画風の種類, strength=適用の強さ, seed=42)

try:
    表示する(run_task('A quiet school library, warm sunlight, simple illustration, no text'))
except Exception as e:
    print('【未実施】画像生成はできませんでした:', type(e).__name__, str(e)[:250])

#### 【見本】6.画像に文字や余白を加える

確かめること：題名と図形を組み合わせた画像が表示されるか。

In [ ]:
# 見本6：文字と枠を配置する。元の図案は「見本用にコードで描いた図形」です。

def フォント(大きさ):
    候補 = ['/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
            '/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc',
            '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf']
    for path in 候補:
        if Path(path).exists(): return ImageFont.truetype(path, 大きさ)
    return ImageFont.load_default()

def run_task(題名):
    if not 題名.strip(): return '題名を入力してください。'
    img = Image.new('RGB', (600,400), '#f1f5f9')
    draw = ImageDraw.Draw(img)
    draw.rectangle((32, 32, 568, 368), fill='#ffffff', outline='#bdcbd8', width=4)
    draw.ellipse((85,125,220,260), fill='#9ad1bf')
    draw.rectangle((265,140,480,245), fill='#f7cf91')
    draw.text((60,52), 題名, fill='#27374d', font=フォント(27))
    return img

表示する(run_task('Notice'))

#### 【見本】7.決まった手順で画像を加工する

確かめること：画像の縦横比を保ち、指定した大きさに揃えられるか。

In [ ]:
# 見本7：写真の縦横比を保って同じ大きさに揃えます。

def run_task(入力):
    try: size = int(入力)
    except ValueError: return '画像サイズを整数で入力してください（例：480）。'
    if not 100 <= size <= 1000:
        return '100〜1000の範囲で指定してください。'
    元画像 = Image.new('RGB', (640,360), '#dcefe7')
    d = ImageDraw.Draw(元画像)
    d.rectangle((80,80,360,290), fill='#4b8979')
    d.ellipse((390,75,555,235), fill='#efaa65')
    return ImageOps.pad(元画像, (size,size), color='white')

表示する(run_task('480'))

### 1-5. ワーク：生成AIに見本を改変してもらう

1-4で動かした見本を、生成AIに自分の課題に合わせて改変してもらいましょう。

＜ノートブックでの作業＞

1. 下の「自分のプロンプト（第10回）」のセルを確認します。
2. ＜ ＞で示した部分を、1-3の内容に書きかえます。使うデータやLoRAがあれば名前も書きます。
3. プロンプトの最後の「見本：」のあとに、1-4で実行した見本のコードを貼り付けます。
4. 生成AIのチャットに送信します。
5. 返ってきたプログラムを「自分のプログラム」に貼り付け、実行して結果を確かめます。

生成AIに画像を送る場合は、顔や氏名などの個人情報が含まれていないか確認してください。

 > **「自分のプロンプト」の記入例**
 >
 > **例1（リユース用の衣類画像）**
 >
 > - 困りごと：リユース用に集めた衣類の画像を、種類別に整理するのに時間がかかる。
 > - まず作りたい機能：画像を1枚入力すると、衣類の種類が表示される。
 > - 使うデータ：Fashion-MNIST（まず商品画像で試す）。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 困りごと：本の紹介カードに合う挿絵を用意するのが大変。
 > - まず作りたい機能：場面を指示すると、水彩画風の挿絵を1枚生成する。
 > - 使うLoRA：水彩画風。

 > **注意**\
 > 個人情報や、許可なく外部へ送ってはいけない画像・資料は、生成AIに送らないようにしましょう。\
 > 必要なら先生が用意したサンプルを使います。

---
**自分のプロンプト（第10回）**<font color="red">＜入力＞</font>

下のセルをダブルクリックして編集しましょう。```で囲まれた文の＜ ＞を、自分の内容に置きかえて使います。

```text
次のPythonプログラム（見本）を、私の課題に合わせて書きかえてください。

困りごと：＜⑤のまとめの一文＞
まず作りたい機能：＜⑩の最小限の機能＞
選んだタイプ：＜1〜7の番号＞
使うデータ・LoRA：＜名前。なければ「なし」＞
試す入力：＜画像番号、指示文など＞
期待する出力：＜予測した種類、画像など＞

条件：
・Google Colabの1つのコードセルで動くプログラムを、全文返してください。
・関数名は run_task のまま、入力は1つにしてください。
・1-2で準備したデータやモデルを使い、準備用のコードは書きかえないでください。
・画像は display などで確認できるようにしてください。
・見本から変えた点を簡単に説明してください。

見本：
＜1-4で実行した見本のコードを貼り付ける＞
```

---
**自分のプログラム（第10回）**<font color="red">＜入力＞</font>

In [ ]:
# ここに自分のプログラムを貼り付けましょう。

---
**比べた結果（第10回）**<font color="red">＜入力＞</font>

- エラーが発生しなかったか：
- 期待される結果になったか：
- プログラムの修正の必要（ある／ない）：

---

 > **ヒント**\
 > **エラーが出たら**：エラーの内容を生成AIに伝え、直したプログラムを全文もらいましょう。\
 > **期待と違ったら**：どの画像・どの指示で、何が違ったかを伝えましょう。 > 修正する場合は下の欄を使います。

#### 手順8〜10（修正が必要な人）

修正が必要な人だけ行いましょう。修正しなくてよい人は空欄のまま、1-6へ進みます。

---
**自分の修正用プロンプト（第10回）**<font color="red">＜入力＞</font>

下の＜ ＞を、実際に試した内容へ書きかえましょう。

```text
今のプログラムで、次の入力を試しました。

試した入力：＜画像番号・指示文など＞
期待した結果：＜　＞
実際の結果：＜　＞
直したいこと：＜　＞

1-2の準備セルは変更せず、run_task の修正版を全文返してください。
```

---
**自分の修正版プログラム（第10回）**<font color="red">＜入力＞</font>

In [ ]:
# 修正したプログラムを貼り付けましょう。

---

### 1-6. ワーク：⑧のテスト例で試す

⑩の入力で期待どおりに動いたら、⑧のテスト例4件でも試してみましょう。

1. 下の「⑧のテスト例」のセルを、⑧で決めた入力に書きかえます。画像認識では画像番号（例：`sample:0`）、画像生成では指示文を使います。
2. その下の4つのセルを実行し、出力を期待する結果と比べます。
3. 「成功・未達・想定外」で判定し、下の欄に記入します。試せなかった場合は「未実施」と記録します。

| 判定 | 意味 |
| :--- | :--- |
| 成功 | 期待した結果になった |
| 未達 | 動いたが、期待した結果にはならなかった |
| 想定外 | 予想していない結果やエラーが出た |
| 未実施 | 実行できなかった |

---
**⑧のテスト例**<font color="red">＜入力＞</font>

In [ ]:
# ⑧ テスト例。選んだタイプの番号だけ変更し、4件の入力を確認してください。
選んだタイプ = 1  # ← 1-1で選んだ番号に変更
例の入力 = {
    1: ['sample:0', 'sample:40', 'sample:70', 'sample:9999'],
    2: ['sample:0', 'sample:40', 'sample:70', 'sample:9999'],
    3: ['sample:0,40,70', 'sample:3,43,73', 'sample:0,0,70', 'sample:9999'],
    4: ['A school library, warm light, no text',
        'Students planting flowers, simple illustration, no text',
        'A school library, winter evening, blue colors, no text', ''],
    5: ['A school library, warm light, watercolor style, no text',
        'Students planting flowers, watercolor style, no text',
        'A school library, winter evening, blue colors, watercolor style', ''],
    6: ['Notice', 'School event', 'A longer announcement title', ''],
    7: ['400', '600', '800', '0'],
}
テスト1, テスト2, テスト3, テスト4 = 例の入力[選んだタイプ]
print('試す入力:', テスト1, '／', テスト2, '／', テスト3, '／', repr(テスト4))
# 自分の課題に合う入力へ書き換えてもかまいません。

---

In [ ]:
# 一般的な入力1（第10回）
# このセルは変更しません
テスト実行(1, '第10回')

In [ ]:
# 一般的な入力2（第10回）
# このセルは変更しません
テスト実行(2, '第10回')

In [ ]:
# 条件を変えた入力（第10回）
# このセルは変更しません
テスト実行(3, '第10回')

In [ ]:
# 想定外の入力（第10回）
# このセルは変更しません
テスト実行(4, '第10回')

 > **「⑧テスト例の結果」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - 一般的な入力1：成功。正解ラベルと同じ種類を予測した。
 > - 一般的な入力2：未達。別の種類を予測した。
 > - 条件を変えた入力：未達。見た目が異なる画像では正解しなかった。
 > - 想定外の入力：想定外。範囲外の画像番号としてエラーが出た。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 一般的な入力1：成功。本が見える挿絵になった。
 > - 条件を変えた入力：未達。指定した色が反映されなかった。

---
**⑧テスト例の結果（第10回）**<font color="red">＜入力＞</font>

- 一般的な入力1：
- 一般的な入力2：
- 条件を変えた入力：
- 想定外の入力：

---

### 1-7. ワーク：開発ログを書く

ここでいう「開発ログ」とは、何をして、どうなったか、次に何をするかを書き残す記録です。\
今回試したことを、下の欄に書きましょう。次回はこの記録をもとに改善します。
⑩が動かなかった場合は、どこで止まったかを「確かめたことと結果」に書きます。

 > **「開発ログ」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - 生成AIに頼んだこと：画像と予測結果を並べて表示するようにしてもらった。
 > - 確かめたことと結果：画像は表示できたが、1枚を間違えて分類した。
 > - 次にすること：間違えた画像を確認する。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 生成AIに頼んだこと：水彩画風の挿絵を作るように変更した。
 > - 確かめたことと結果：本は描かれたが背景が目立ちすぎた。
 > - 次にすること：本が目立つ指示に変える。

---
**開発ログ（第10回）**<font color="red">＜入力＞</font>

- 生成AIに頼んだこと：
- 確かめたことと結果：
- 次にすること：

---

## まとめ（第10回）

今回の演習では、次の内容を学びました。

- ひな型（見本）を生成AIに改変してもらい、自分の課題に合わせられる。
- エラーや期待と違う結果が出たら、何が違ったのかを伝えて直してもらう。
- ⑩の入力だけでなく、⑧のテスト例でも試すと、改善点が見つかる。

### 次回への接続

次回（第11回）は、今回の結果を見て改善するところを1つ選びます。問題がなければ、修正不要と考えた理由を記録します。\
このノートブックは次回も使うので、保存しておきましょう。

## 2. 結果を見て改善する（第11回）

第10回の結果をもとに、必要なら改善するところを1つ選びます。修正が不要でも、確かめた結果を残しましょう。\
第10回と同じ入力で改善前後を比べ、ペアの人にも確かめてもらいます。

### 2-1. 実行：前回の続きから始める

ノートブックを開き直したときは、前回の出力が表示されていても、プログラムの準備が必要なことがあります。

次の順にセルを実行しましょう。

1. 1-2の準備セル（選んだタイプに必要なもの）
2. 「自分のプログラム（第10回）」のセル（修正版があれば、修正版のセル）
3. 「⑧のテスト例」のセル

画像生成では、モデルを再び読み込むことがあります。

### 2-2. ワーク：改善するところを1つ決める

第10回の「⑧の結果」と開発ログの「次にすること」を見て、改善するところを1つ決めましょう。

| 第10回の状態 | 今回やること |
| :--- | :--- |
| ⑩がまだ動いていない | 1-5の続きをして、まず⑩を動かす |
| ⑧で試せていない入力がある | 改善前のプログラムで試し、結果を記録する |
| 期待と違う結果があった | 改善するところを1つ選ぶ |
| すべて期待どおり | 別の入力で確認し、修正不要とした理由を書く |

画像認識では誤分類した画像、画像生成では満たせなかった条件を手がかりにします。

 > **「改善するところ」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - 試した入力：`sample:40`
 > - 期待する出力：正しい衣類の種類が表示される。
 > - 実際の出力：違う種類が表示された。
 > - 直したいこと：学習する画像の枚数を増やして試す。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 試した入力：図書室の挿絵を生成する指示文。
 > - 期待する出力：本が目立つ挿絵。
 > - 実際の出力：背景ばかりが目立つ。
 > - 直したいこと：本を手前に描くよう指示を変える。

---
**改善するところ**<font color="red">＜入力＞</font>

- 試した入力：
- 期待する出力：
- 実際の出力：
- 直したいこと：

---

### 2-3. ワーク：生成AIに直してもらう

2-2で直すところが決まったら、今のプログラムと一緒に生成AIのチャットに送りましょう。修正が不要なら2-4を飛ばして2-5へ進みます。

＜ノートブックでの作業＞

1. 「自分のプロンプト（第11回）」の＜ ＞を書きかえます。
2. 最後の「今のプログラム：」に第10回で使ったプログラムを貼り付けます。
3. 生成AIに送信し、返ってきたコードを「自分のプログラム（第11回・改善後）」へ貼り付けます。
4. 同じ入力で実行して結果を確認します。

---
**自分のプロンプト（第11回）**<font color="red">＜入力＞</font>

下のセルをダブルクリックするとセルが編集できます。編集状態にした後、下のセルの```で囲まれている中にプロンプトを書きましょう。
その際、元々目印として入れている＜＞で囲われている部分は消し、置き換える形で書き換えましょう。

```text
次のPythonプログラムを直してください。

直したいこと：＜　＞
試した入力：＜　＞
期待する出力：＜　＞
実際の出力：＜　＞

条件：
・Google Colabのコードセルで動くプログラムを全文返してください。
・関数名は run_task のままにしてください。
・1-2の準備セルは変更しないでください。
・画像を出力する場合は画面に表示できるようにしてください。

今のプログラム：
＜第10回の自分のプログラムを貼り付ける＞
```

---
**自分のプログラム（第11回・改善後）**<font color="red">＜入力＞</font>

In [ ]:
# 改善したプログラムを貼り付けましょう。

---

 > **ヒント**\
 > エラーが出たら表示された内容を生成AIへ伝えましょう。\
 > 画像認識なら予測と正解の違い、画像生成なら指示した条件と実際の画像の違いを伝えます。

---
**確認した結果（第11回）**<font color="red">＜入力＞</font>

- エラーが発生しなかったか：

- 期待される出力になったか：

- プログラムの修正の必要（ある/ない）：

---

#### 手順8〜10（修正が必要な人）

修正が必要な人だけ行いましょう。必要がなければ空欄のまま2-4へ進みます。

---
**自分の修正用プロンプト（第11回）**<font color="red">＜入力＞</font>

下の＜ ＞を、自分の結果に合わせて書きかえましょう。

```text
今のプログラムで試した結果：＜　＞
期待した結果：＜　＞
直したいこと：＜　＞

準備セルは変更せず、run_task の修正版を全文返してください。
```

---
**自分の修正版プログラム（第11回）**<font color="red">＜入力＞</font>

In [ ]:
# 必要なら修正したプログラムを貼り付けましょう。

---

### 2-4. ワーク：同じテスト例の入力で改善前と比べる

改善した場合は、改善後のプログラムで、第10回と同じ⑧のテスト例を試します。\
第10回の結果は上書きせず、記録を見ながら比べましょう。

直した部分だけでなく、もともとうまくいっていた部分が変わっていないかも確かめましょう。画像生成では、比較する画像の乱数設定もできるだけ揃えます。

In [ ]:
# 一般的な入力1（第11回）
# このセルは変更しません
テスト実行(1, '第11回')

In [ ]:
# 一般的な入力2（第11回）
# このセルは変更しません
テスト実行(2, '第11回')

In [ ]:
# 条件を変えた入力（第11回）
# このセルは変更しません
テスト実行(3, '第11回')

In [ ]:
# 想定外の入力（第11回）
# このセルは変更しません
テスト実行(4, '第11回')

 > **「改善前と改善後の比較」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - よくなったこと：前回間違えた画像を正しく分類した。
 > - 変わってしまったこと：別の画像を間違えて分類するようになった。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - よくなったこと：本が手前に描かれるようになった。
 > - 変わってしまったこと：背景の色も変わった。

---
**改善前と改善後の比較**<font color="red">＜入力＞</font>

- よくなったこと：
- 変わってしまったこと（あれば）：

---

### 2-5. 確認：ペアで確かめる

ペアの人と交代で、作ったものを見せ合いましょう。

- **見せる人**：困りごとと、試した入力・結果を説明します。改善した場合は前後の結果を見せます。
- **見る人**：期待した結果と実際の結果を比べ、よい点と気になる点を1つずつ伝えます。

気になる点があれば、別の入力でも試してみましょう。

---
**ペアの質問を試す**<font color="red">＜入力＞</font>

In [ ]:
# ペアと試す追加の入力。タイプに応じた文字列へ書き換えます。
追加の入力 = 'sample:3'  # CNNの例。画像生成なら英語の指示文など。
try:
    表示する(run_task(追加の入力))
except Exception as e:
    print('【未実施・想定外】', type(e).__name__, str(e)[:300])

---

---
**ペアに言われたこと**<font color="red">＜入力＞</font>

- ペアに言われたこと：
- 試して分かったこと（あれば）：

---

### 2-6. ワーク：開発ログを書く

今回試したことを、開発ログに書きましょう。変更が不要なら「修正不要」と書き、確かめた結果を残します。\
次回は、このログを見て検証するプログラムを決めます。

 > **「開発ログ」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - 生成AIに頼んだこと：学習画像の数を増やして試すように直してもらった。
 > - 確かめたことと結果：間違えていた画像を正しく予測できた。
 > - 次にすること：ほかの画像でも正しく予測できるか確かめる。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 生成AIに頼んだこと：本が手前に見えるよう指示を変えた。
 > - 確かめたことと結果：本が目立つようになった。
 > - 次にすること：ほかの本でも使えるか確かめる。

---
**開発ログ（第11回）**<font color="red">＜入力＞</font>

- 生成AIに頼んだこと：
- 確かめたことと結果：
- 次にすること：

---

## まとめ（第11回）

今回の演習では、次の内容を学びました。

- 期待する結果と実際の結果を比べ、必要なら改善点を1つ決める。
- 改善前と改善後は、同じ入力で比べる。
- 直した点だけでなく、他の結果が変わっていないかも確かめる。

### 次回への接続

次回（第12回）は、確かめるプログラムを決め、⑧のテスト例で動作を確かめて結果を記録します。\
このノートブックは次回も使うので、保存しておきましょう。

## 3. 動作を確かめて記録する（第12回）

ここでは、確かめるプログラムを決め、⑧のテスト例で結果を記録します。\
⑦の成功指標について分かったことを整理し、他の人に使ってもらう画面を用意します。

### 3-1. 実行：前回の続きから始める

次の順に実行しましょう。

1. 1-2の準備セル（選んだタイプに必要なもの）
2. いちばん新しい「自分のプログラム」のセル
3. 「⑧のテスト例」のセル

### 3-2. ワーク：確かめるプログラムを決める

今回確かめるプログラムを、下の「確かめるプログラム（第12回）」に残しましょう。このプログラムを「検証版」と呼びます。

まず、第11回の開発ログを見て、今日直すことがあれば記入します。

- 直すことがなければ、いちばんよく動いているプログラムを検証版へ貼り付けます。
- 直すことがあれば、修正して動作を確かめてから検証版へ貼り付けます。

**テストが終わるまでは、検証版の内容を書きかえないでください。**

 > **「今日直すこと」の記入例**
 >
 > - 例1（衣類の種類を予測する）：入力番号が範囲外のとき、分かりやすい案内を出す。
 > - 例2（図書委員会の挿絵）：指示文が空欄なら「入力してください」と表示する。

---
**今日直すこと**<font color="red">＜入力＞</font>

- 直すこと（あれば）：

---
**今日直したこと**<font color="red">＜入力＞</font>

- 直したこと（あれば）：

---
**確かめるプログラム（第12回）**<font color="red">＜入力＞</font>

In [ ]:
# 第12回で確かめるプログラムを貼り付けましょう。

---

### 3-3. ワーク：⑧のテスト例で確かめる

下の4つのセルを実行し、⑧で決めた期待する結果と比べて判定しましょう。\
判定は第10回と同じく「成功・未達・想定外・未実施」です。

テスト中に直したい点が見つかっても、検証版は書きかえずに、まず結果を記録します。直したいことは3-7の開発ログに書きましょう。

In [ ]:
# 一般的な入力1（第12回）
# このセルは変更しません
テスト実行(1, '第12回')

In [ ]:
# 一般的な入力2（第12回）
# このセルは変更しません
テスト実行(2, '第12回')

In [ ]:
# 条件を変えた入力（第12回）
# このセルは変更しません
テスト実行(3, '第12回')

In [ ]:
# 想定外の入力（第12回）
# このセルは変更しません
テスト実行(4, '第12回')

 > **「⑧テスト例での結果」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - 一般的な入力1：成功。正解ラベルと同じ種類を予測した。
 > - 一般的な入力2：未達。別の種類を予測した。
 > - 条件を変えた入力：未達。予測した種類が正解と違った。
 > - 想定外の入力：想定外。存在しない画像番号でエラーが出た。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 一般的な入力1：成功。本のある図書室が描かれた。
 > - 条件を変えた入力：未達。指定した色が十分に反映されなかった。

---
**⑧テスト例での結果（第12回）**<font color="red">＜入力＞</font>

- 一般的な入力1：
- 一般的な入力2：
- 条件を変えた入力：
- 想定外の入力：

---

### 3-4. ワーク：⑦の成功指標について整理する

⑧のテスト例で分かるのは、プログラムが期待どおりに動くかどうかです。\
⑦の成功指標は、困りごとが解決したかどうかを確かめるためのものです。⑧だけでは分からないこともあります。\
今の時点で分かったことと、まだ確かめていないことを整理しましょう。

画像認識の正解率を調べる場合は、4件だけでなく、学習に使っていない複数のテスト画像を使います。

 > **「⑦成功指標についての整理」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - 分かったこと：テスト画像に対して予測ができた。
 > - まだ確かめていないこと：実際に衣類を整理する時間が短くなるか。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 分かったこと：指定した雰囲気の画像を作れた。
 > - まだ確かめていないこと：紹介カードを作る人が使いやすいか。

---
**⑦の成功指標についての整理**<font color="red">＜入力＞</font>

- 分かったこと：
- まだ確かめていないこと：

---

### 3-5. ワーク：他の人が使える画面を用意する

第13回からは、作ったものを他の人に使ってもらいます。プログラムを知らない人でも使えるように、入力欄と［実行］ボタンがある画面を用意しましょう。

1. 下のセルの「題名」と「説明」を自分の課題に合わせて書きかえます。
2. 実行すると、題名・説明・入力欄・［実行］ボタンが表示されます。
3. 入力欄に⑧の一般的な入力1を入れ、結果を確かめます。

 > **「使ってもらうための画面」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - 題名：衣類の種類を調べる
 > - 説明：画像番号を入力して［実行］を押してください。（例：sample:0）
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 題名：おすすめ本の挿絵づくり
 > - 説明：描きたい場面を英語で入力し、［実行］を押してください。

---
**使ってもらうための画面**<font color="red">＜入力＞</font>

In [ ]:
# 使ってもらうための画面
題名 = '画像を使った課題解決の試作品'
説明 = '画像番号（例 sample:0）、指示文、またはサイズを入力してください。'

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    入力欄 = widgets.Text(description='入力', placeholder='sample:0', layout=widgets.Layout(width='85%'))
    実行ボタン = widgets.Button(description='実行', button_style='info')
    表示欄 = widgets.Output()
    def 押したとき(_):
        with 表示欄:
            clear_output(wait=True)
            try:
                表示する(run_task(入力欄.value))
            except Exception as e:
                print('実行できませんでした:', type(e).__name__, str(e)[:300])
    実行ボタン.on_click(押したとき)
    display(widgets.HTML(f'<h3>{題名}</h3><p>{説明}</p>'))
    display(入力欄, 実行ボタン, 表示欄)
except ImportError:
    print('この環境にはipywidgetsがありません。入力欄の代わりに以下で試してください。')
    print('  表示する(run_task("sample:0"))')

---

### 3-6. 確認：ペアで確かめる

ペアの人と交代で、結果を見せ合いましょう。

- **見せる人**：困りごとと、⑧の入力・期待する結果・実際の結果を説明します。
- **見る人**：結果と判定を比べ、気になることがあれば質問します。

実際に試したい場合は3-5の画面で入力します。

---
**ペアに言われたこと（第12回）**<font color="red">＜入力＞</font>

- ペアに言われたこと：
- 直した判定（あれば）：

---

### 3-7. ワーク：開発ログを書く

今回やったことを開発ログに書きましょう。テストで見つかった改善点も「次に直すこと」に残します。\
第13回からは、他の人に使ってもらいながら改善を続けます。

**次回に使うもの**：1-3の課題、1-6・3-3のテスト結果、第10回のプログラム、今回の検証版が残っているか確認しましょう。

 > **「開発ログ」の記入例**
 >
 > **例1（衣類の種類を予測する）**
 >
 > - 変えたこと（あれば）：画像番号の入力方法を画面に表示した。
 > - 次に直すこと：他の人が自分の写真を入力できるか検討する。
 >
 > **例2（図書委員会の挿絵）**
 >
 > - 変えたこと（あれば）：指示文が空欄なら案内を出すようにした。
 > - 次に直すこと：図書委員に試してもらい、挿絵が使いやすいか確かめる。

---
**開発ログ（第12回）**<font color="red">＜入力＞</font>

- 変えたこと（あれば）：
- 次に直すこと：

---

## まとめ（第12回）

今回の演習では、次の内容を学びました。

- どのプログラムで確かめたか分かるように、検証版を決めてからテストする。
- ⑧のテスト結果と、⑦の成功指標（困りごとが解決したか）は分けて考える。
- 他の人が使うためには、入力欄や［実行］ボタンのある画面が必要になる。

### 次回への接続

次回（第13回）からは、このノートブックで作ったプログラムと画面を使い、他の人に使ってもらいながら改善を続けます。\
第10〜12回の記録が残っていることを確認し、ノートブックを保存して提出しましょう。